In [ ]:
import json
import pandas as pd
import spacy

In [ ]:
# Load SpaCy natural language model
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Dictionaries for each column to store unique IDs
user_ids = {}
item_ids = {}
aspect_ids = {}
opinion_ids = {}

In [ ]:
# Independent counters to assign id to each column
user_id_counter = 1
item_id_counter = 1000000
opinion_id_counter = 2000000
aspect_id_counter = 3000000

In [ ]:
# Assign a unique id to each column
def get_unique_id(word, id_dict, id_counter):
    word_lower = word.lower()
    if word_lower not in id_dict:
        id_dict[word_lower] = id_counter
        id_counter += 1
    return id_dict[word_lower], id_counter

In [ ]:
# List to store processed data
processed_data = []

In [ ]:
# Input dataset path
# input_file_path = '/content/drive/MyDrive/Datasets/Digital_Music-Copy.jsonl'
input_file_path = '/content/drive/MyDrive/Datasets/Digital_Music.jsonl'

# Output dataset path
output_file_path = '/content/drive/MyDrive/Datasets/processed_reviews2.inter'

In [ ]:
# Read the data
with open(input_file_path, 'r') as f:
    for line in f:
        review = json.loads(line)

        # Finding columns dynamically
        user = review.get("user_id") or review.get("user") or review.get("userid")
        asin = review.get("asin") or review.get("item_id") or review.get("product_id") or review.get("itemid") or review.get("productid")
        rating = review.get("rating") or review.get("score") or review.get("grade")
        text = review.get("text") or review.get("review") or review.get("comment") or review.get("content")

        if rating is not None:
            rating = float(rating)

        # Assign unique ID to user and item
        user_id, user_id_counter = get_unique_id(user, user_ids, user_id_counter)
        item_id, item_id_counter = get_unique_id(asin, item_ids, item_id_counter)

        #Text processing with NLP
        if text is None:
          text = ""  # Replace None with an empty string
        doc = nlp(text)

        aspects = []
        opinions = []
        aspect_id_list = []
        opinion_id_list = []
        tag_list = []

        # Extraction of aspects & opinions based on dependency parsing
        for token in doc:
            if token.dep_ in ["nsubj", "nmod", "dobj"] and token.pos_ == "NOUN":
                aspect_id, aspect_id_counter = get_unique_id(token.text, aspect_ids, aspect_id_counter)
                aspects.append(token.text.lower())
                aspect_id_list.append(aspect_id)
            elif token.dep_ in ["amod", "acomp"] and token.pos_ == "ADJ":
                opinion_id, opinion_id_counter = get_unique_id(token.text, opinion_ids, opinion_id_counter)
                opinions.append(token.text.lower())
                opinion_id_list.append(opinion_id)

        # Match opinions with aspects and create the tag column
        for aspect, opinion in zip(aspects, opinions):
            # tag_list.append(f"{aspect} {opinion}")
            tag_list.append(f"{opinion} {aspect}")

        tag_token_seq = opinion_id_list + aspect_id_list
        tag_token_seq_str = ",".join(map(str, tag_token_seq))

        # Add the processed data as a list in a line
        processed_data.append({
            "user_id:token": user_id,
            "item_id:token": item_id,
            "rating:float": rating,
            "opinion_id:token": ",".join(map(str, opinion_id_list)),
            "opinion:tag": ",".join(opinions),
            "aspect_id:token": ",".join(map(str, aspect_id_list)),
            "aspect:tag": ",".join(aspects),
            # "tag:token_seq": tag_token_seq_str,
            "tag:token_seq": ",".join(tag_list),  # New combined tag column
            "review": text
        })

In [ ]:
# Save dataset in .inter format
df = pd.DataFrame(processed_data)
df.to_csv(output_file_path, index=False, sep='\t')